# Phase 3.5 - Statistical Latent Geometry

Esta fase testa se ha sinal estatisticamente robusto de corretude no espaco latente antes de qualquer steering causal agressivo.

Configuramos caminhos e carregamos as tabelas geradas por `scripts/analyze_latent_geometry.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
OUT = ROOT / 'runs' / 'phase35' / 'latent_geometry'
OUT

A funcao abaixo torna o notebook robusto: se uma tabela ainda nao existir, ela retorna um DataFrame vazio.

In [ ]:
def read_csv(name):
    path = OUT / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

probe = read_csv('probe_results.csv')
latent = read_csv('latent_score_results.csv')
perm = read_csv('permutation_results.csv')
reg = read_csv('regression_results.csv')
surv = read_csv('survival_results.csv')
boot = read_csv('bootstrap_results.csv')
summary = json.loads((OUT / 'summary.json').read_text(encoding='utf-8')) if (OUT / 'summary.json').exists() else {}
summary

## Parte A - Dataset latente

Resumo do numero de tarefas, tentativas, classes e configuracoes presentes nas ativacoes extraidas.

In [ ]:
if latent.empty:
    print('Sem scores latentes ainda. Rode scripts/analyze_latent_geometry.py depois da Fase 3.')
else:
    dataset_summary = pd.Series({
        'tasks': latent['task_id'].nunique() if 'task_id' in latent else None,
        'attempt_rows': latent[['task_id', 'attempt']].drop_duplicates().shape[0] if {'task_id','attempt'}.issubset(latent.columns) else len(latent),
        'layers': latent['layer'].nunique(),
        'correct_rate': latent.drop_duplicates(['sample_index', 'layer'])['label'].mean() if 'sample_index' in latent else latent['label'].mean(),
        'directions': latent['direction_type'].nunique(),
    })
    display(dataset_summary.to_frame('value'))

Distribuicoes por dificuldade e modelo ajudam a detectar vieses de composicao antes de interpretar probes.

In [ ]:
if latent.empty:
    print('Sem dados.')
else:
    cols = [c for c in ['difficulty', 'difficulty_class', 'model_id', 'direction_type'] if c in latent.columns]
    for col in cols:
        display(latent[col].value_counts(dropna=False).to_frame('count').head(10))

## Parte B - Separabilidade por camada

Comparamos AUC, accuracy e Brier score dos probes lineares por camada.

In [ ]:
probe.sort_values('auc', ascending=False).head(10) if not probe.empty else pd.DataFrame()

In [ ]:
if probe.empty:
    print('Sem resultados de probe.')
else:
    ax = probe.pivot_table(index='layer', columns='probe_type', values='auc', aggfunc='max').plot(marker='o')
    ax.set_ylabel('AUC')
    ax.set_title('Probe AUC por camada')
    plt.show()

## Parte C - Controles estatisticos

Permutation tests verificam se o melhor score poderia surgir por acaso sob labels embaralhados.

In [ ]:
perm.sort_values('p_value').head(20) if not perm.empty else pd.DataFrame()

Compare a direcao de corretude com controles negativo, aleatorio, labels embaralhados e comprimento.

In [ ]:
if latent.empty:
    print('Sem scores latentes.')
else:
    control_auc = latent.groupby(['layer','direction_type'])['score_auc_for_layer'].mean().reset_index()
    display(control_auc.sort_values('score_auc_for_layer', ascending=False).head(20))

## Parte D - Score latente

A distribuicao do score deve separar respostas corretas e incorretas se a direcao capturar sinal real.

In [ ]:
if latent.empty:
    print('Sem scores.')
else:
    best_layer = summary.get('best_probe_layer') or latent['layer'].iloc[0]
    focus = latent[(latent['layer'] == best_layer) & (latent['direction_type'] == 'correctness_direction')]
    ax = focus.boxplot(column='latent_score', by='label')
    ax.set_title(f'Latent score por classe - layer {best_layer}')
    ax.set_xlabel('label: 0 incorreta, 1 correta')
    ax.set_ylabel('latent_score')
    plt.suptitle('')
    plt.show()

## Parte E - Regressao de sucesso

A regressao compara modelos sem score latente, somente score latente e observaveis + score latente.

In [ ]:
reg.sort_values('auc', ascending=False) if not reg.empty else pd.DataFrame()

## Parte F - Survival analysis

Aqui Best-of-N vira tempo ate primeira resposta correta, com censura quando a tarefa nao foi resolvida no orcamento.

In [ ]:
if surv.empty:
    print('Sem runs da Fase 2 para survival analysis.')
else:
    km = surv[surv['analysis'] == 'kaplan_meier']
    for run_label, group in km.groupby('run_label'):
        plt.plot(group['attempt'], group['cumulative_resolution'], marker='o', label=run_label)
    plt.xlabel('Tentativa k')
    plt.ylabel('Probabilidade acumulada de resolver')
    plt.title('Resolucao acumulada Best-of-N')
    plt.legend()
    plt.show()

In [ ]:
surv[surv['analysis'] == 'summary'].sort_values('mean_attempts').head(10) if not surv.empty else pd.DataFrame()

## Parte G - Sintese para TCC

Esta tabela resume a melhor camada candidata, significancia por permutacao e ganho incremental do score latente.

In [ ]:
final_table = pd.DataFrame([{
    'best_probe_layer': summary.get('best_probe_layer'),
    'best_probe_auc': summary.get('best_probe_auc'),
    'permutation_p_value': summary.get('permutation_p_value'),
    'latent_score_delta_auc': summary.get('latent_score_delta_auc'),
    'best_survival_config': summary.get('best_survival_config'),
    'mean_attempt_reduction_candidate': summary.get('mean_attempt_reduction_candidate'),
    'recommended_steering_layers': summary.get('recommended_steering_layers'),
}])
final_table

Conclusao cautelosa: se houver poucas tarefas, trate os resultados como piloto. A Fase 3.5 identifica hipoteses estatisticas; a Fase 4 testa causalidade com controles.